# HCDE 530 — Week 5: Merging Two Tables with `pd.merge()`

Sometimes the data you need is split across two files. A merge combines them on a shared column — the same idea as a JOIN in SQL.

This demo uses two small tables:
- **`reviews_mini.csv`** — 20 app reviews, each tagged with an `app_id`
- **`app_info.csv`** — one row per app, with the full name, category, company, and founding year

Neither table alone answers "what is the average rating for each category?" Together, they can.

## Setup

Run the next code cell first. It installs pandas into the active notebook kernel if needed, then imports it.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## Load the two tables

The agent only needs to know the filename and what you want to call the DataFrame. Give it the filename exactly as it appears on disk.

Ask the agent:
>"Load the file `reviews_mini.csv` into a DataFrame called `reviews` using pandas. Print the shape and show the first few rows."

In [2]:
reviews = pd.read_csv('reviews_mini.csv')
print(f"reviews: {reviews.shape[0]} rows, {reviews.shape[1]} columns")
reviews

reviews: 20 rows, 4 columns


,review_id,app_id,rating,review_text
0,1,FLD,5,Remote fieldwork tracking finally works. Batte...
1,2,FLD,2,GPS accuracy drops in dense urban areas. Not u...
2,3,FLD,4,Offline sync is solid. Would like better expor...
3,4,FLD,3,Good concept but the UI needs work. Too many t...
4,5,LBK,5,The highlight reel feature saved hours of anal...
5,6,LBK,4,Participant recruiting add-on is overpriced bu...
6,7,LBK,1,Recording failed mid-session. Lost 40 minutes ...
7,8,LBK,3,Works well but video quality degrades on slow ...
8,9,DVT,5,Best research repository I have used. Tagging ...
9,10,DVT,4,Import from Notion is seamless. Search could b...


In [3]:
apps = pd.read_csv('app_info.csv')
print(f"apps: {apps.shape[0]} rows, {apps.shape[1]} columns")
apps

apps: 5 rows, 5 columns


,app_id,app_name,category,company,year_founded
0,FLD,Fieldkit,field research,Fieldkit Inc.,2018
1,LBK,Lookback,user research,Lookback AB,2013
2,DVT,Dovetail,research repository,Dovetail Research,2017
3,MZE,Maze,usability testing,Maze Design,2018
4,MRO,Miro,collaborative whiteboard,RealtimeBoard,2011


## The problem

`reviews` has `app_id` — a short code (`FLD`, `LBK`, etc.). It does not have the full app name or category.

`apps` has the full name and category, but no review data.

To answer "average rating by category," you need both. That's what a merge does.

## Merge on the shared column — `pd.merge()`

For the agent to write the merge correctly, it needs three things:
1. The names of both DataFrames
2. The column they share (the column name must be identical in both tables)
3. Whether to keep only matching rows (inner — the default) or all rows from the left table (left)

A strong prompt lists all three explicitly. A weak prompt says "merge the tables" — the agent will guess at the column name and may get it wrong.

Ask the agent:
>"I have two DataFrames called `reviews` and `apps`. `reviews` has columns: review_id, app_id, rating, review_text. `apps` has columns: app_id, app_name, category, company, year_founded. Merge them on `app_id` so that every review row gets the matching app information added to it. Keep only rows where `app_id` appears in both tables. Call the result `merged` and show the first 10 rows."

In [4]:
merged = reviews.merge(apps, on='app_id')

print(f"merged: {merged.shape[0]} rows, {merged.shape[1]} columns")
merged.head(10)

merged: 20 rows, 8 columns


,review_id,app_id,rating,review_text,app_name,category,company,year_founded
0,1,FLD,5,Remote fieldwork tracking finally works. Batte...,Fieldkit,field research,Fieldkit Inc.,2018
1,2,FLD,2,GPS accuracy drops in dense urban areas. Not u...,Fieldkit,field research,Fieldkit Inc.,2018
2,3,FLD,4,Offline sync is solid. Would like better expor...,Fieldkit,field research,Fieldkit Inc.,2018
3,4,FLD,3,Good concept but the UI needs work. Too many t...,Fieldkit,field research,Fieldkit Inc.,2018
4,5,LBK,5,The highlight reel feature saved hours of anal...,Lookback,user research,Lookback AB,2013
5,6,LBK,4,Participant recruiting add-on is overpriced bu...,Lookback,user research,Lookback AB,2013
6,7,LBK,1,Recording failed mid-session. Lost 40 minutes ...,Lookback,user research,Lookback AB,2013
7,8,LBK,3,Works well but video quality degrades on slow ...,Lookback,user research,Lookback AB,2013
8,9,DVT,5,Best research repository I have used. Tagging ...,Dovetail,research repository,Dovetail Research,2017
9,10,DVT,4,Import from Notion is seamless. Search could b...,Dovetail,research repository,Dovetail Research,2017


The result has all the columns from both tables — one row per review, now with the app's full name, category, and company attached.

Notice the row count: 20 reviews in, 20 rows out. Every `app_id` in `reviews` had a match in `apps`. When there is no match, rows are dropped (that is the default — an inner join).

## Now answer the question that required both tables

Once the merge is done, the combined DataFrame is just a regular DataFrame — you can run any of the operations from the pandas demo on it.

Ask the agent:
>"Using the DataFrame `merged`, which has columns: review_id, app_id, rating, review_text, app_name, category, company, year_founded — show me the average rating and review count for each category, sorted from lowest to highest average rating."

In [5]:
# Average rating per category — only possible after the merge
merged.groupby('category')['rating'].agg(['mean', 'count']).round(2).sort_values('mean')

,mean,count
category,,
user research,3.25,4
field research,3.50,4
usability testing,3.50,4
research repository,4.00,4
collaborative whiteboard,4.25,4


Ask the agent:
>"Using the DataFrame `merged`, show me the average rating per app_name, sorted from lowest to highest. Round to 2 decimal places."

In [6]:
# Average rating per app — now you can show app_name instead of the cryptic app_id
merged.groupby('app_name')['rating'].mean().round(2).sort_values()

app_name
Lookback    3.25
Fieldkit    3.50
Maze        3.50
Dovetail    4.00
Miro        4.25
Name: rating, dtype: float64

## When to use a merge

Use `pd.merge()` when:
- Your dataset has an ID column that points to information in another file
- You want to add descriptive labels (names, categories, metadata) to a table of events or measurements
- You are joining survey responses to a participant list

The typical pattern in an MP1 notebook:

```python
# df is your main dataset; metadata is a lookup table
combined = df.merge(metadata, on='shared_column_name')
```

The prompt pattern that works every time — paste both column lists and be explicit:

>"I have two DataFrames. The first is called `[name]` with columns: [list them]. The second is called `[name]` with columns: [list them]. Merge them on `[shared column]`. Keep all rows from the first table even if there is no match in the second."

## A note on join types

By default, `.merge()` does an **inner join** — only rows with a match in both tables are kept. Other options:

| `how=` | Keeps |
|--------|-------|
| `'inner'` (default) | Only rows that match in both tables |
| `'left'` | All rows from the left table; NaN where no match in right |
| `'right'` | All rows from the right table; NaN where no match in left |
| `'outer'` | All rows from both tables; NaN wherever there is no match |

For most HCD data work, `inner` or `left` is what you need. Tell the agent which one when you ask for the merge — or ask it to explain the difference first.